# 🤖 Stock Alert — Ollama Server on Google Colab

รัน Ollama บน Colab GPU ฟรี แล้วเปิด tunnel ให้แอป Stock Alert PWA เรียกใช้ AI วิเคราะห์หุ้น

**ขั้นตอน:** รันทุก cell ตามลำดับ → คัดลอก URL `https://xxxx.trycloudflare.com` จาก cell สุดท้าย → ไปใส่ในหน้า **Settings → AI** ของแอป

⚠️ Colab เซสชันหมดอายุ (ไม่ได้ใช้ ~90 นาที / สูงสุด 12 ชม.) เมื่อเซสชันดับ URL จะเปลี่ยน ต้องรันใหม่และอัปเดต URL ในแอป

In [ ]:
# 1) Install Ollama + cloudflared
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print('✅ installed')

In [ ]:
# 2) Start Ollama server in background
import subprocess, time
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('✅ ollama serve started (pid', ollama_proc.pid, ')')

In [ ]:
# 3) Pull a model (T4 GPU มี RAM ~15GB → ใช้โมเดล 7-8B ได้สบาย)
MODEL = 'llama3.1:8b'   # ทางเลือก: 'qwen2.5:7b' (ไทยค่อนข้างดี), 'llama3.2:3b' (เร็ว)
!ollama pull {MODEL}
!ollama list

In [ ]:
# 4) Quick smoke test
import json, urllib.request
def ask(prompt):
    req = urllib.request.Request('http://localhost:11434/api/generate',
        data=json.dumps({'model': MODEL, 'prompt': prompt, 'stream': False, 'format': 'json'}).encode(),
        headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())['response']
print(ask('ตอบ JSON: {"ok": true, "msg": "สวัสดี"}'))

In [ ]:
# 5) Expose public tunnel + keep alive loop
import threading, time, subprocess, urllib.request

def keepalive():
    while True:
        try:
            urllib.request.urlopen('http://localhost:11434/api/tags', timeout=10)
        except Exception:
            pass
        time.sleep(60)

threading.Thread(target=keepalive, daemon=True).start()

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

import re
url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = proc.stdout.readline()
    if not line and proc.poll() is not None:
        break
    m = re.search(r'https://\S*trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

if url:
    print('=' * 60)
    print('🔗 Ollama URL ของคุณ:', url)
    print('   → คัดลอกไปใส่ในหน้า Settings → AI ในแอป Stock Alert')
    print('=' * 60)
    # รักษา tunnel ไว้
    try:
        while True:
            time.sleep(60)
            if proc.poll() is not None:
                print('tunnel หลุด รีสตาร์ต...')
                proc = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
                                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except KeyboardInterrupt:
        proc.terminate()
else:
    print('❌ สร้าง tunnel ไม่สำเร็จ ลองรัน cell นี้ใหม่')